In [292]:
# install libraries
!pip install pandas numpy matplotlib seaborn scikit-learn

In [293]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [294]:
# Load the dataset
data = pd.read_csv(r'C:\Users\woro_\OneDrive\DS_Projects\ML_Fraud_Detection_System\data\nova_pay_combined.csv')

In [295]:
# Explore the variable columns in the dataset.
pd.set_option('display.max_columns', None)
data.head()

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,device_id,new_device,ip_address,ip_country,location_mismatch,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,1.351351,9f292dcc-3297-4947-a260-6a1ef69041ff,False,221.78.171.180,US,False,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,12.758621,3a95b9f5-309f-4684-a46d-e2ff2435bf78,True,120.12.20.29,CA,False,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,7.142857,a4737752-9aac-43ed-9d8b-2ccdffc24052,False,223.96.181.93,US,False,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,0.925926,6aeb85a3-5603-4221-896c-9e6882764f1a,False,186.228.15.74,US,False,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,83.333333,a5b9250e-dbe0-4c5f-a6e7-5492b7349402,False,11.82.47.62,US,False,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [296]:
# Explore the distribution of the target variable. Examine the skewness of the data and determine if there is a class imbalance issue.
# In this case there is a class imbalance.
data['is_fraud'].value_counts()

is_fraud
0    10403
1      997
Name: count, dtype: int64

In [297]:
# Check for missing values in the dataset and examine the data types of each variable to identify any potential issues
# data type mismatch.
pd.reset_option('display.max_rows')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  str    
 1   customer_id                11400 non-null  str    
 2   timestamp                  11371 non-null  str    
 3   home_country               11400 non-null  str    
 4   source_currency            11400 non-null  str    
 5   dest_currency              11400 non-null  str    
 6   channel                    11400 non-null  str    
 7   amount_src                 11400 non-null  str    
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  str    
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  str    
 14  i

In [298]:
# DEALING WITH DUPLICATE ROWS
# Check for duplicate rows in the dataset and determine how to handle them (e.g., remove)
# ── EXPLORE DATA──────────────────────────────────────────────────────────────────

number_dup = data.duplicated().sum()
transaction_id_dup = data["transaction_id"].duplicated().sum()
print(f"Fully duplicate rows        : {number_dup}")
print(f"Duplicate transaction_id    : {transaction_id_dup}")
print(f"Unique transaction_ids      : {data['transaction_id'].nunique():,} / {len(data):,}")

# Show a pair of duplicates
example_dup = data[data["transaction_id"].duplicated(keep=False)]["transaction_id"].iloc[0]
print(f"\nExample duplicate pair (transaction_id = {example_dup}):")
print(data[data["transaction_id"] == example_dup][
    ["transaction_id", "amount_usd", "channel", "is_fraud"]
].to_string(index=True))


Fully duplicate rows        : 200
Duplicate transaction_id    : 200
Unique transaction_ids      : 11,200 / 11,400

Example duplicate pair (transaction_id = 4662cb2d-21b2-4390-ba38-5a1eddebdc7c):
                             transaction_id  amount_usd channel  is_fraud
57     4662cb2d-21b2-4390-ba38-5a1eddebdc7c      152.75  mobile         0
10030  4662cb2d-21b2-4390-ba38-5a1eddebdc7c      152.75  mobile         0


In [299]:
# ── FIX ──────────────────────────────────────────────────────────────────────
data.drop_duplicates(subset="transaction_id", keep="first", inplace=True)
data.reset_index(drop=True, inplace=True)
print(f"\n✔  After duplicate dropping      : {len(data):,} rows")


✔  After duplicate dropping      : 11,200 rows


In [300]:
# DEALING WITH NULL VALUES
# Check for missing values in the dataset and examine the data types of each variable to identify any
# ── EXPLORE DATA──────────────────────────────────────────────────────────────────
null_counts = data.isnull().sum()
null_pct    = (null_counts / len(data) * 100).round(2)
null_report = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_report = null_report[null_report["null_count"] > 0].sort_values("null_count", ascending=False)
print("\nColumns with missing values:")
print(null_report.to_string())

# Check if nulls cluster in the same rows (likely one data-feed failure)
cols_with_nulls = null_report.index.tolist()
any_null_mask = data[cols_with_nulls].isnull().any(axis=1)
print(f"\nRows where at least one column is null  : {any_null_mask.sum()}")
print(f"Row-wise null overlap (all null cols)   : "
      f"{data[cols_with_nulls].isnull().all(axis=1).sum()}")


Columns with missing values:
                    null_count  null_pct
amount_usd                 300      2.68
ip_address                 300      2.68
ip_country                 296      2.64
kyc_tier                   295      2.63
fee                        290      2.59
device_trust_score         290      2.59
timestamp                   29      0.26

Rows where at least one column is null  : 329
Row-wise null overlap (all null cols)   : 0


In [301]:
# ── FIX ──────────────────────────────────────────────────────────────────────
# Rows missing EITHER critical financial field → drop
critical_cols = ["amount_usd", "fee"]
before = len(data)
data.dropna(subset=critical_cols, how="any", inplace=True)
print(f"\n✔  Dropped rows missing amount_usd OR fee : {before - len(data)}")

# ip_address / ip_country — fill with sentinel (PII often absent legitimately)
data["ip_address"]  = data["ip_address"].fillna("unknown")
data["ip_country"]  = data["ip_country"].fillna("unknown")
print(f"✔  ip_address / ip_country nulls filled with 'unknown'")

# kyc_tier — fill with mode
kyc_mode = data["kyc_tier"].mode()[0]
data["kyc_tier"] = data["kyc_tier"].fillna(kyc_mode)
print(f"✔  kyc_tier nulls filled with mode ('{kyc_mode}')")

# device_trust_score — fill with median (score column, skewed possible)
dts_median = data["device_trust_score"].median()
data["device_trust_score"] = data["device_trust_score"].fillna(dts_median)
print(f"✔  device_trust_score nulls filled with median ({dts_median:.3f})")

# timestamp — already handled in Challenge 6
remaining_nulls = data.isnull().sum().sum()
print(f"\n✔  Remaining nulls (excl. timestamp) : {remaining_nulls}")


✔  Dropped rows missing amount_usd OR fee : 300
✔  ip_address / ip_country nulls filled with 'unknown'
✔  kyc_tier nulls filled with mode ('standard')
✔  device_trust_score nulls filled with median (0.658)

✔  Remaining nulls (excl. timestamp) : 29


In [302]:
# DEALING WITH INCONSISTENT CATEGORICAL VARIABLES
# # ── EXPLORE DATA──────────────────────────────────────────────────────────────────
CAT_COLS = ["channel", "kyc_tier", "home_country", "ip_country",
            "source_currency", "dest_currency"]

for col in CAT_COLS:
    vc = data[col].value_counts(dropna=False)
    # flag non-canonical values: those with spaces, mixed case, typos
    print(f"\n{col}  ({data[col].nunique(dropna=False)} unique):")
    print(vc.head(12).to_string())


channel  (12 unique):
channel
mobile       6054
web          3595
ATM           961
mobille        58
 mobile        47
MOBILE         45
unknown        36
WEB            34
 web           33
weeb           24
ATm             9
 ATM            4

kyc_tier  (12 unique):
kyc_tier
standard       7780
enhanced       1794
low            1034
standrd          72
STANDARD         69
 standard        64
unknown          29
 enhanced        17
ENHANCED         16
enhancd          12
 low              8
LOW               5

home_country  (7 unique):
home_country
US         7578
UK         2025
CA         1182
 US          63
unknown      32
 UK          13
 CA           7

ip_country  (7 unique):
ip_country
US         6713
UK         2372
CA         1703
 US          48
unknown      31
 UK          18
 CA          15

source_currency  (3 unique):
source_currency
USD    7664
GBP    2045
CAD    1191

dest_currency  (9 unique):
dest_currency
NGN    1405
USD    1270
INR    1269
CNY    1267
PHP    1

In [303]:
# ── FIX ──────────────────────────────────────────────────────────────────────
def normalise_category(series: pd.Series,
                       canonical_map: dict,
                       default: str = "unknown") -> pd.Series:
    """
    Strip whitespace, uppercase, then map to canonical values.
    Anything not in canonical_map → default.
    """
    s = series.astype(str).str.strip().str.upper()
    return s.map(canonical_map).fillna(default)

# channel canonical map
CHANNEL_MAP = {
    "MOBILE": "mobile", "MOBILLE": "mobile",
    "WEB":    "web",    "WEEB":    "web",
    "ATM":    "atm",
    "UNKNOWN": "unknown",
}

# kyc_tier canonical map
KYC_MAP = {
    "STANDARD": "standard", "STANDRD": "standard",
    "ENHANCED": "enhanced",
    "LOW":      "low",
    "UNKNOWN":  "unknown",
}

# home / ip country canonical map
COUNTRY_MAP = {
    "US": "US", "UK": "UK", "CA": "CA",
    "UNKNOWN": "unknown", "NAN": "unknown", "NaN": "unknown",
}

data["channel"]    = normalise_category(data["channel"],     CHANNEL_MAP)
data["kyc_tier"]   = normalise_category(data["kyc_tier"],    KYC_MAP)
data["home_country"] = normalise_category(data["home_country"], COUNTRY_MAP)
data["ip_country"] = normalise_category(data["ip_country"],  COUNTRY_MAP)

# source_currency / dest_currency are already upper-case codes — just strip spaces
for col in ["source_currency", "dest_currency"]:
    data[col] = data[col].astype(str).str.strip().str.upper()

print("\n✔  Post-normalisation value counts:")
for col in CAT_COLS:
    print(f"  {col}: {dict(data[col].value_counts())}")



✔  Post-normalisation value counts:
  channel: {'mobile': np.int64(6204), 'web': np.int64(3686), 'atm': np.int64(974), 'unknown': np.int64(36)}
  kyc_tier: {'standard': np.int64(7985), 'enhanced': np.int64(1827), 'low': np.int64(1047), 'unknown': np.int64(41)}
  home_country: {'US': np.int64(7641), 'UK': np.int64(2038), 'CA': np.int64(1189), 'unknown': np.int64(32)}
  ip_country: {'US': np.int64(6761), 'UK': np.int64(2390), 'CA': np.int64(1718), 'unknown': np.int64(31)}
  source_currency: {'USD': np.int64(7664), 'GBP': np.int64(2045), 'CAD': np.int64(1191)}
  dest_currency: {'NGN': np.int64(1405), 'USD': np.int64(1270), 'INR': np.int64(1269), 'CNY': np.int64(1267), 'PHP': np.int64(1264), 'GBP': np.int64(1234), 'CAD': np.int64(1227), 'EUR': np.int64(1139), 'MXN': np.int64(825)}


In [304]:
#FINAL NAME — ENCODING FOR CATEGORICALS VARIABLES
# Mapped old ordinal order (kyc_tier) which was not clear, to the new ordinal order (kyc_order) and created a new column (kyc_tier_encoded).

# Ordinal encode kyc_tier
kyc_order = {"low": 0, "standard": 1, "enhanced": 2, "unknown": -1}
data["kyc_tier_encoded"] = data["kyc_tier"].map(kyc_order)
data.drop(columns=["kyc_tier"], inplace=True)
print("✔  kyc_tier ordinal-encoded (low=0, standard=1, enhanced=2, unknown=-1)")

# One-hot encode remaining categoricals
ohe_cols = ["channel", "home_country", "ip_country",
            "source_currency", "dest_currency"]
data = pd.get_dummies(data, columns=ohe_cols, drop_first=False, dtype="uint8")
print(f"✔  One-hot encoded: {ohe_cols}")

# ip_address_hash — drop if present (was only for audit; leakage risk if kept)
data.drop(columns=["ip_address_hash"], inplace=True, errors="ignore")
print("✔  Dropped ip_address_hash (high-cardinality, leakage risk)")

# Boolean columns already bool — ensure uint8 for sklearn compat
for col in ["new_device", "location_mismatch"]:
    data[col] = data[col].astype("uint8")
print("✔  new_device, location_mismatch cast to uint8")

✔  kyc_tier ordinal-encoded (low=0, standard=1, enhanced=2, unknown=-1)
✔  One-hot encoded: ['channel', 'home_country', 'ip_country', 'source_currency', 'dest_currency']
✔  Dropped ip_address_hash (high-cardinality, leakage risk)
✔  new_device, location_mismatch cast to uint8


In [305]:
# DROP IDENTIFIER COLUMNS (They have no predictive power and they cause overfitting)
# ── EXPLORE DATA──────────────────────────────────────────────────────────────────
id_cols = ["transaction_id", "customer_id", "device_id", "ip_address"]
for col in id_cols:
    print(f"  {col:<20} unique={data[col].nunique():,}  dtype={data[col].dtype}")

# Check if transaction_id has any remaining duplicates
print(f"\n  Remaining duplicate transaction_ids : {data['transaction_id'].duplicated().sum()}")

  transaction_id       unique=10,900  dtype=str
  customer_id          unique=1,314  dtype=str
  device_id            unique=2,112  dtype=str
  ip_address           unique=10,900  dtype=str

  Remaining duplicate transaction_ids : 0


In [306]:
# ── FIX ──────────────────────────────────────────────────────────────────────
# transaction_id: unique row key — drop as ML feature (no predictive value)
data.drop(columns=["transaction_id"], inplace=True)
print("\n✔  Dropped transaction_id  (unique row key, no predictive signal)")

# ip_address: PII — hash to anonymise (keeps grouping ability)
import hashlib
data["ip_address_hash"] = data["ip_address"].apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()[:12] if x != "unknown" else "unknown"
)
data.drop(columns=["ip_address"], inplace=True)
print("✔  ip_address hashed to ip_address_hash (SHA-256 prefix, PII removed)")

# customer_id / device_id: encode as integer IDs (frequency encoding)
for col in ["customer_id", "device_id"]:
    freq = data[col].value_counts().to_dict()
    data[f"{col}_freq"] = data[col].map(freq)
    data.drop(columns=[col], inplace=True)
    print(f"✔  {col} replaced with {col}_freq (frequency encoding)")


✔  Dropped transaction_id  (unique row key, no predictive signal)
✔  ip_address hashed to ip_address_hash (SHA-256 prefix, PII removed)
✔  customer_id replaced with customer_id_freq (frequency encoding)
✔  device_id replaced with device_id_freq (frequency encoding)


In [307]:
# INCORRECT TIMESTAMP FORMAT
# ── EXPLORE ──────────────────────────────────────────────────────────────────
nat_count = data["timestamp"].isna().sum()
print(f"NaT (unparseable / originally null) timestamps : {nat_count}")

NaT (unparseable / originally null) timestamps : 29


In [308]:
# ── FIX ──────────────────────────────────────────────────────────────────────
# Ensure timestamp is datetime-like before extracting features
data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")

# Derive useful ML features from timestamp before dropping NaTs
data["hour_of_day"]   = data["timestamp"].dt.hour
data["day_of_week"]   = data["timestamp"].dt.dayofweek    # 0=Mon, 6=Sun
data["is_weekend"]    = data["day_of_week"].isin([5, 6]).astype("uint8")
print(f"\n✔  Engineered: hour_of_day, day_of_week, is_weekend")

# Drop rows with invalid timestamps (they cannot be recovered)
before = len(data)
data.dropna(subset=["timestamp"], inplace=True)
print(f"✔  Dropped {before - len(data)} rows with NaT timestamps")
data.drop(columns=["timestamp"], inplace=True)
print(f"✔  Dropped raw timestamp column (features extracted)")



✔  Engineered: hour_of_day, day_of_week, is_weekend
✔  Dropped 60 rows with NaT timestamps
✔  Dropped raw timestamp column (features extracted)


In [ ]:
# INVALID NUMERIC VALUES
# ── EXPLORE ──────────────────────────────────────────────────────────────────
numeric_check_cols = ["amount_src", "amount_usd", "fee", "device_trust_score", "ip_risk_score", "txn_velocity_1h"]

for col in numeric_check_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

checks = {
    "amount_src < 0"          : (data["amount_src"] < 0).sum(),
    "amount_usd < 0"          : (data["amount_usd"] < 0).sum(),
    "fee < 0"                 : (data["fee"] < 0).sum(),
    "fee > 1000 (outlier)"    : (data["fee"] > 1000).sum(),
    "device_trust_score < 0"  : (data["device_trust_score"] < 0).sum(),
    "ip_risk_score > 1"       : (data["ip_risk_score"] > 1).sum(),
    "txn_velocity_1h < 0"     : (data["txn_velocity_1h"] < 0).sum(),
}
for desc, count in checks.items():
    print(f"  {desc:<35} : {count}")

# Additional: amount_src has abs-mismatch with amount_usd
data["_abs_src"] = data["amount_src"].abs()
# For USD source, amount_src == amount_usd (exchange rate = 1)
if "source_currency" in data.columns:
    usd_rows = data[data["source_currency"] == "USD"].copy()
else:
    usd_rows = data[data["source_currency_USD"] == 1].copy()
mismatch = (usd_rows["_abs_src"].round(2) != usd_rows["amount_usd"].round(2)).sum()
print(f"  USD rows where |amount_src| ≠ amount_usd : {mismatch}")
data.drop(columns=["_abs_src"], inplace=True)

  amount_src < 0                      : 97
  amount_usd < 0                      : 0
  fee < 0                             : 93
  fee > 1000 (outlier)                : 97
  device_trust_score < 0              : 190
  ip_risk_score > 1                   : 190
  txn_velocity_1h < 0                 : 190


KeyError: 'source_currency'

In [ ]:
# ── FIX ──────────────────────────────────────────────────────────────────────
# amount_src: take absolute value (sign error in source system)
neg_src_count = (data["amount_src"] < 0).sum()
data["amount_src"] = data["amount_src"].abs()
print(f"\n✔  amount_src: flipped {neg_src_count} negatives to absolute value")

# fee: negative fees are clearly erroneous — clip to 0
neg_fee_count = (data["fee"] < 0).sum()
data["fee"] = data["fee"].clip(lower=0)
print(f"✔  fee: clipped {neg_fee_count} negative values to 0")

# fee > 1000: likely data-entry errors — cap at 99th percentile
fee_p99 = data["fee"].quantile(0.99)
extreme_fee = (data["fee"] > fee_p99).sum()
data["fee"] = data["fee"].clip(upper=fee_p99)
print(f"✔  fee: capped {extreme_fee} values above 99th percentile ({fee_p99:.2f})")

# device_trust_score: must be [0, 1] — clip
neg_dts = (data["device_trust_score"] < 0).sum()
data["device_trust_score"] = data["device_trust_score"].clip(0, 1)
print(f"✔  device_trust_score: clipped {neg_dts} negatives to 0")

# ip_risk_score: must be [0, 1] — clip
over_irs = (data["ip_risk_score"] > 1).sum()
data["ip_risk_score"] = data["ip_risk_score"].clip(0, 1)
print(f"✔  ip_risk_score: clipped {over_irs} values > 1 to 1.0")

# txn_velocity_1h: velocity cannot be negative — clip to 0
neg_vel = (data["txn_velocity_1h"] < 0).sum()
data["txn_velocity_1h"] = data["txn_velocity_1h"].clip(lower=0)
print(f"✔  txn_velocity_1h: clipped {neg_vel} negatives to 0")


KeyError: 'transaction_velocity_1h'